# ADAN8888 — Week 8: Develop Third Modeling Approach

## ===== WEEK 8 START =====

**Goal:** Forecast next-month passenger demand for U.S. domestic carrier–route–month observations from official BTS T-100 data, using a third modeling family: **feedforward multilayer perceptron (MLP) regression**.

**Continuity:** The exact Week 3 chronological split and Week 5 training-fitted PCA features (`PC1`, `PC2`) are preserved. Week 6 Ridge and Week 7 HGB_C provide historical validation benchmarks. No test targets are scored or used for selection.

**Assignment design:** Train **three** neural architectures; report training and validation MAE, RMSE and R² (passenger counts); choose the Week 8 winner by validation MAE. Where exact ties occur, use validation RMSE and fewer trainable weights. A lower validation score is better.

**Code architecture:** All modeling functions found in `Src/week8_src_mlp_neural_network.py`.

**Important:** 
- Week 8 is a separate third model family, and more architecture complexity does **not** guarantee outperforming Week 7. 
- Al true metrics must be generated locally from the original data.

## 1. Load Week 8 code and identify project paths

In [ ]:
from pathlib import Path
import importlib.util
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

def find_root():
    here = Path.cwd().resolve()
    for location in (here,*here.parents):
        if (location / "Src" / "week8_src_mlp_neural_network.py").is_file():
            return location
    raise FileNotFoundError("Open the existing repository root or Notebooks folder.")

ROOT = find_root()
SOURCE = ROOT / "Src" / "week8_src_mlp_neural_network.py"
spec = importlib.util.spec_from_file_location("week8_mlp",SOURCE)
week8 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = week8
spec.loader.exec_module(week8)
REPORTS = ROOT / "Reports" / "week8"
TABLES = REPORTS / "week8_tables"
FIGURES = REPORTS / "week8_figures"
print("Project root:", ROOT)
print("Week 8 reusable code:", SOURCE)
print("Candidate models:", len(week8.CANDIDATES))

## 2. Reconstruct full PC1/PC2 training and validation data

In [ ]:
data, split_summary, source_description = week8.load_full_training_validation(ROOT)
display(split_summary)
print("Input provenance:", source_description)
assert set(data) == {"Training", "Validation"}
assert len(data["Training"]) == 1_062_603
assert len(data["Validation"]) == 239_395
assert pd.to_datetime(data["Training"].ForecastMonth).max() < pd.to_datetime(data["Validation"].ForecastMonth).min()
print("Verified exact temporal separation and full-size datasets.")

## 3. Third modeling approach: neural networks and scaling

A feedforward neural network learns nonlinear transformations with trained weights in multiple hidden layers. `MLPRegressor` uses Adamm optimization and nonlinear ReLU activations; compared with Week 7 boosted trees, it is a distinct modeling family with multiple layers and non-convex training.

| Model | Hidden-layer architecture | Controlled comparison |
|:---|:---|:---|
| MLP_A | (32,16) | Smaller two-hidden-layer starting point |
| MLP_B | (64,32) | Wider layers than A |
| MLP_C | (64,32,16) | Adds a third hidden layer beyond B |

- Shared settings: `alpha=0.001`, `solver='adam'`, `learning_rate_init=0.001`, `batch_size=4096`, `max_iter=45`, `random_state=42`, `early_stopping=False`. 
- Solver can stop using **training loss alone**. 
- No random validation holdout is used.
___
- PCA features standardized on training only in a `Pipeline`. 
- Training target values standardized in a `TransformedTargetRegressor` and predictions are returned to the original passenger scale before MAE, RMSE, and R². 
- All negative predictions clipped to zero (matchs prior-week reporting).

In [ ]:
experiment = pd.DataFrame([
    {"Model":c["Model"], "HiddenLayers":"-".join(map(str,c["hidden_layer_sizes"])),
     "TrainableParameters":week8.parameter_count(c["hidden_layer_sizes"])}
    for c in week8.CANDIDATES
])
display(experiment)
print("Fixed hyperparameters:",week8.COMMON)
assert len(experiment)==3
assert list(experiment.TrainableParameters)==sorted(experiment.TrainableParameters.tolist())

## 4. Fit all three candidate neural networks

Each candidate fits on the **full training partition** and is evaluated separately using the full training and subsequent validation partitions.

In [ ]:
result = week8.fit_three_variations(
    data["Training"], data["Validation"], split_summary, source_description
)
display(result.comparison.round(4))
assert len(result.comparison)==3
assert len(result.metrics_long)==6

## 5. Analyze generalization, optimize against validation MAE and select the winner

- A more complex architecture could reduce training loss potentially and worsening validation error.
- However, A simple validation-minus-training gap cannot by itself establish overfitting because time periods can differ in traffic mix. 
- We therefore compare all **three** MAE, RMSE, and R² outcomes and acknowledge trade-offs across metrics.

In [ ]:
display(result.ranking[["Model","HiddenLayers","TrainableParameters","FittedEpochs",
                        "Training_MAE","Validation_MAE","Training_RMSE","Validation_RMSE",
                        "Training_R2","Validation_R2","MAE_Gap_ValidationMinusTraining"]].round(4))
print("Validation-MAE-selected Week 8 winner:",result.selected_name)
print("Chosen architecture:",result.settings["hidden_layer_sizes"])
print("Week 7 HGB_C validation MAE benchmark:",f"{week8.WEEK7_VALIDATION_MAE_REFERENCE:,.2f}","passengers")
assert result.selected_name==result.ranking.iloc[0].Model

## 6. Save reproducible results

In [ ]:
week8.save_results(result,ROOT)
print((REPORTS/"week8_verified_results_for_report.md").read_text(encoding="utf8"))
print("Models:",[x.name for x in (ROOT/"Models"/"week8").glob("*")])

## 7. Review metric, prediction, neural-optimization figures

- MAE/RMSE comparisons and monthly validation MAE use the whole dataset. 
- Actual-versus-predicted scatter displays 4,000 reproducibly sampled rows only; all numerical metrics use full partition s. 
- Optimization loss is on the **scaled target** (not passenger-count MAE).

In [ ]:
for filename in [
    "week8_figure_mae_comparison.png", "week8_figure_rmse_comparison.png",
    "week8_figure_validation_actual_vs_predicted.png",
    "week8_figure_monthly_validation_mae.png", "week8_figure_training_loss_curves.png"
]:
    image_file = FIGURES / filename
    assert image_file.is_file(),f"Missing figure {image_file}"
    print(filename)
    display(Image(filename=str(image_file)))

## 8. Final audit and end marker

In [ ]:
assert len(pd.read_csv(TABLES/"week8_three_model_comparison.csv"))==3
assert len(pd.read_csv(TABLES/"week8_metrics_long.csv"))==6
assert (TABLES/"week8_winner_summary.json").is_file()
assert (ROOT/"Models"/"week8"/"week8_selected_mlp_neural_network.joblib").is_file()
for col in ("Training_MAE","Validation_MAE","Training_RMSE","Validation_RMSE","Training_R2","Validation_R2"):
    assert np.isfinite(result.comparison[col]).all()
assert result.selected_name in {"MLP_A","MLP_B","MLP_C"}
print("All Week 8 checks passed.")
print("===== WEEK 8 END =====")